# 제조 데이터 전반 EDA

공정 센서, 품질 검사, 설비 가동률 데이터를 개별 진단하고 공통 기간에서만 안전하게 비교합니다. 원본 Excel 파일은 수정하지 않습니다.

> **연계 제약:** 센서 데이터는 2026-03-02부터, 가동률 데이터는 2026-03-01까지이므로 두 데이터를 직접 결합하지 않습니다.

In [ ]:
from pathlib import Path
import warnings
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

DATA_DIR = Path.cwd().parent / 'data'
if not DATA_DIR.exists():
    DATA_DIR = Path('data')  # 노트북을 프로젝트 루트에서 실행하는 경우

sensor = pd.read_excel(DATA_DIR / '01_process_sensor.xlsx')
quality = pd.read_excel(DATA_DIR / '02_quality_inspection.xlsx')
uptime = pd.read_excel(DATA_DIR / '03_machine_uptime.xlsx')

sensor['timestamp'] = pd.to_datetime(sensor['timestamp'])
sensor['date'] = sensor['timestamp'].dt.normalize()
quality['inspect_date'] = pd.to_datetime(quality['inspect_date'])
uptime['date'] = pd.to_datetime(uptime['date'])
uptime['line_id'] = uptime['machine_id'].str.split('-').str[0]

print(f'데이터 경로: {DATA_DIR.resolve()}')

## 1. 데이터 개요 및 품질 점검

In [ ]:
datasets = {'공정 센서': sensor, '품질 검사': quality, '설비 가동률': uptime}
overview = []
for name, df in datasets.items():
    date_col = 'timestamp' if name == '공정 센서' else ('inspect_date' if name == '품질 검사' else 'date')
    overview.append({
        '데이터': name, '행': len(df), '열': df.shape[1], '중복 행': int(df.duplicated().sum()),
        '시작일': df[date_col].min(), '종료일': df[date_col].max(),
        '결측 셀': int(df.isna().sum().sum())
    })
pd.DataFrame(overview)

In [ ]:
missing = pd.concat({name: df.isna().sum() for name, df in datasets.items()}, axis=1).fillna(0).astype(int)
display(missing[missing.sum(axis=1) > 0])

print('센서 설비 수:', sensor['machine_id'].nunique(), '| 가동률 설비 수:', uptime['machine_id'].nunique())
print('공통 설비:', sorted(set(sensor['machine_id']) & set(uptime['machine_id'])))
print('공통 라인:', sorted(set(sensor['line_id']) & set(quality['line_id']) & set(uptime['line_id'])))

## 2. 공정 센서 탐색

In [ ]:
sensor_metrics = ['temp_C', 'pressure_bar', 'vibration_mm_s', 'humidity_pct', 'cycle_time_sec']
display(sensor[sensor_metrics + ['defect_flag']].describe().T)
display(sensor.groupby(['line_id', 'shift']).agg(관측수=('machine_id', 'size'), 불량률=('defect_flag', 'mean')).round(3))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, metric in zip(axes.flat, sensor_metrics):
    sns.boxplot(data=sensor, x='line_id', y=metric, hue='line_id', legend=False, ax=ax)
    ax.set_title(f'라인별 {metric}')
axes.flat[-1].axis('off')
plt.tight_layout()

In [ ]:
daily_sensor = sensor.groupby(['date', 'line_id'], as_index=False).agg(
    관측수=('machine_id', 'size'), 센서불량률=('defect_flag', 'mean'), 평균온도=('temp_C', 'mean'), 평균사이클타임=('cycle_time_sec', 'mean')
)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.lineplot(data=daily_sensor, x='date', y='센서불량률', hue='line_id', marker='o', ax=axes[0])
sns.lineplot(data=daily_sensor, x='date', y='평균사이클타임', hue='line_id', marker='o', ax=axes[1])
axes[0].set_title('일자·라인별 센서 불량 플래그 비율')
axes[1].set_title('일자·라인별 평균 사이클타임')
plt.tight_layout()

plt.figure(figsize=(8, 6))
sns.heatmap(sensor[sensor_metrics + ['defect_flag']].corr(), annot=True, fmt='.2f', cmap='vlag', center=0)
plt.title('센서 변수와 불량 플래그 상관관계')
plt.show()

## 3. 품질 검사 탐색

In [ ]:
quality['defect_rate'] = quality['defect_qty'] / quality['insp_qty']
quality_summary = quality.groupby('line_id').agg(로트수=('lot_id', 'size'), 검사수량=('insp_qty', 'sum'), 불량수량=('defect_qty', 'sum')).reset_index()
quality_summary['불량률'] = quality_summary['불량수량'] / quality_summary['검사수량']
display(quality_summary.round(3))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.boxplot(data=quality, x='line_id', y='defect_rate', hue='line_id', legend=False, ax=axes[0])
sns.barplot(data=quality_summary, x='line_id', y='불량률', hue='line_id', legend=False, ax=axes[1])
top_defects = quality['defect_type'].fillna('미분류').value_counts().head(10).reset_index()
top_defects.columns = ['불량유형', '건수']
sns.barplot(data=top_defects, y='불량유형', x='건수', ax=axes[2])
axes[0].set_title('라인별 로트 불량률')
axes[1].set_title('라인별 가중 불량률')
axes[2].set_title('상위 불량 유형')
plt.tight_layout()

In [ ]:
quality_daily = quality.groupby(['inspect_date', 'line_id'], as_index=False).agg(검사수량=('insp_qty', 'sum'), 불량수량=('defect_qty', 'sum'))
quality_daily['품질불량률'] = quality_daily['불량수량'] / quality_daily['검사수량']
plt.figure(figsize=(13, 4))
sns.lineplot(data=quality_daily, x='inspect_date', y='품질불량률', hue='line_id', marker='o', linewidth=1)
plt.title('일자·라인별 품질 검사 불량률')
plt.tight_layout()
plt.show()

display(quality.groupby('supplier')['defect_rate'].agg(['count', 'mean', 'median']).sort_values('mean', ascending=False).round(3))

## 4. 설비 가동률 탐색

In [ ]:
uptime['availability'] = uptime['run_min'] / uptime['plan_min']
uptime['downtime_rate'] = uptime['downtime_min'] / uptime['plan_min']
uptime['yield_rate'] = uptime['good_qty'] / uptime['prod_qty']
uptime['energy_per_unit'] = uptime['energy_kWh'] / uptime['prod_qty']
line_uptime = uptime.groupby('line_id').agg(가동가능률=('availability', 'mean'), 비가동률=('downtime_rate', 'mean'), 양품률=('yield_rate', 'mean'), 생산량=('prod_qty', 'sum'), 에너지_kWh=('energy_kWh', 'sum'))
display(line_uptime.round(3))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.boxplot(data=uptime, x='line_id', y='availability', hue='line_id', legend=False, ax=axes[0])
sns.boxplot(data=uptime, x='line_id', y='yield_rate', hue='line_id', legend=False, ax=axes[1])
reasons = uptime.groupby('downtime_reason', dropna=False)['downtime_min'].sum().sort_values(ascending=False).head(10).reset_index()
sns.barplot(data=reasons, y='downtime_reason', x='downtime_min', ax=axes[2])
axes[0].set_title('라인별 가동 가능률')
axes[1].set_title('라인별 양품률')
axes[2].set_title('비가동 사유별 시간 합계')
plt.tight_layout()

## 5. 공통 기간의 라인·일자 연계 분석

In [ ]:
uptime_daily = uptime.groupby(['date', 'line_id'], as_index=False).agg(계획시간=('plan_min', 'sum'), 가동시간=('run_min', 'sum'), 생산량=('prod_qty', 'sum'), 양품수량=('good_qty', 'sum'))
uptime_daily['가동가능률'] = uptime_daily['가동시간'] / uptime_daily['계획시간']
uptime_daily['양품률'] = uptime_daily['양품수량'] / uptime_daily['생산량']

sensor_quality = daily_sensor.merge(quality_daily, left_on=['date', 'line_id'], right_on=['inspect_date', 'line_id'], how='inner')
uptime_quality = uptime_daily.merge(quality_daily, left_on=['date', 'line_id'], right_on=['inspect_date', 'line_id'], how='inner')

print(f'센서-품질 매칭: {len(sensor_quality)}개 라인·일자 그룹')
print(f'가동률-품질 매칭: {len(uptime_quality)}개 라인·일자 그룹')
print('센서-가동률 직접 매칭: 0개 (분석 기간 비중첩)')

relations = pd.DataFrame({
    '비교': ['센서 불량 플래그 vs 품질 불량률', '가동 가능률 vs 품질 불량률', '양품률 vs 품질 불량률'],
    '매칭 그룹': [len(sensor_quality), len(uptime_quality), len(uptime_quality)],
    '피어슨 상관계수': [
        sensor_quality['센서불량률'].corr(sensor_quality['품질불량률']),
        uptime_quality['가동가능률'].corr(uptime_quality['품질불량률']),
        uptime_quality['양품률'].corr(uptime_quality['품질불량률'])
    ]
})
display(relations.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.regplot(data=sensor_quality, x='센서불량률', y='품질불량률', ax=axes[0], scatter_kws={'s': 55})
sns.regplot(data=uptime_quality, x='가동가능률', y='품질불량률', ax=axes[1], scatter_kws={'s': 35})
axes[0].set_title('센서 플래그와 품질 불량률')
axes[1].set_title('가동 가능률과 품질 불량률')
plt.tight_layout()

## 6. 해석 및 다음 단계

- 이 노트북의 상관계수는 동일 라인·일자에 집계한 **탐색적 지표**이며 인과관계를 의미하지 않습니다.
- 센서와 가동률을 함께 설명변수로 쓰려면 2026-03-02 이후의 가동률 또는 2026-03-01 이전의 센서 데이터가 추가로 필요합니다.
- 다음 단계에서는 로트와 생산 설비를 연결하는 키(예: lot_id, 생산시각, machine_id)를 확보해 공정 조건과 검사 결과를 더 정밀하게 연결합니다.
- 결측 센서값과 비가동 사유의 결측은 원인별로 구분한 뒤, 모델링 전 별도 처리 정책을 정합니다.